# Laboratorio 05: Chatbot de Becas de Posgrado en el Perú

Chatbot **híbrido** con cuatro componentes:

| # | Componente | Técnica | Rol |
|---|-----------|---------|-----|
| 1 | **Reglas** | Expresiones regulares / palabras clave | Saludos, despedidas, listado de becas, contacto |
| 2 | **NLP – Embeddings** | `sentence-transformers` + similitud coseno | Búsqueda semántica sobre la base de conocimiento |
| 3 | **RAG** | Recuperación + prompt con contexto | Respuestas ancladas en los documentos |
| 4 | **LLM local** | **Ollama** (`qwen2.5:1.5b`, ~1 GB) | Redacta la respuesta final en lenguaje natural |

>  **Todas las becas, montos, fechas y requisitos de este notebook son INVENTADOS** con fines académicos. No corresponden a programas reales.

**Flujo:**
```
Pregunta ─► [Reglas] ──(coincide)──► Respuesta fija
               │ no
               ▼
        [Embeddings + coseno] ─► top-k fragmentos ─► ¿score ≥ umbral?
                                                       │ sí            │ no
                                                       ▼               ▼
                                              [RAG + LLM Ollama]   "No tengo información"
```

### Requisitos previos
1. Instalar [Ollama](https://ollama.com/download) y descargar el modelo (una sola vez), en una terminal:
   ```
   ollama pull qwen2.5:1.5b
   ```
   Alternativas: `qwen2.5:0.5b` (aún más liviano, menos preciso) o `llama3.2:3b`. Nota: `llama3.2:1b` se probó y ignora el contexto con frecuencia.
2. Dependencias de Python (siguiente celda).

In [1]:
%pip install -q sentence-transformers numpy requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 0. Configuración

In [2]:
import re
import unicodedata
import numpy as np
import requests

OLLAMA_URL   = "http://localhost:11434"
LLM_MODEL    = "qwen2.5:1.5b"  # ~1 GB; buen español. Alternativa: "llama3.2:3b"
EMB_MODEL    = "paraphrase-multilingual-MiniLM-L12-v2"  # ~120 MB, multilingüe
TOP_K        = 3
UMBRAL       = 0.30            # similitud coseno mínima para considerar relevante un fragmento

## 1. Base de conocimiento (textos inventados)

Siete becas ficticias de posgrado y cuatro documentos generales. Cada fragmento es una unidad de recuperación (*chunk*).

In [3]:
KB = [
    # ---------- Beca Horizonte Perú ----------
    {"id": "horizonte-1", "beca": "Beca Horizonte Perú",
     "texto": "La Beca Horizonte Perú financia estudios de maestría en universidades del extranjero ubicadas en el top 200 de un ranking internacional. Cubre matrícula, pensión, pasaje de ida y vuelta, seguro médico y una subvención mensual de 1,800 dólares. La duración máxima es de 24 meses."},
    {"id": "horizonte-2", "beca": "Beca Horizonte Perú",
     "texto": "Requisitos de la Beca Horizonte Perú: ser peruano, tener grado de bachiller con promedio ponderado mínimo de 14 sobre 20, contar con carta de admisión de la universidad de destino, certificado de inglés nivel B2 y no superar los 35 años de edad. Los becarios deben retornar al Perú y trabajar 2 años por cada año de beca."},
    {"id": "horizonte-3", "beca": "Beca Horizonte Perú",
     "texto": "La convocatoria de la Beca Horizonte Perú abre cada 15 de marzo y cierra el 30 de mayo. Los resultados se publican el 20 de julio en el portal del Instituto Nacional de Becas y Crédito Educativo Andino (INBECA)."},

    # ---------- Beca Amazonía Investigadora ----------
    {"id": "amazonia-1", "beca": "Beca Amazonía Investigadora",
     "texto": "La Beca Amazonía Investigadora está dirigida a estudiantes de doctorado que investiguen biodiversidad, cambio climático o gestión de bosques amazónicos. Financia hasta 4 años de estudios en universidades peruanas o en el extranjero, con una subvención mensual de 2,200 soles y un fondo de investigación anual de 6,000 soles."},
    {"id": "amazonia-2", "beca": "Beca Amazonía Investigadora",
     "texto": "Para postular a la Beca Amazonía Investigadora se requiere grado de magíster, un proyecto de tesis aprobado por un asesor, una carta de respaldo de un centro de investigación y haber publicado al menos un artículo en una revista indexada. Se otorgan puntos adicionales a postulantes de comunidades de Loreto, Ucayali y Madre de Dios."},

    # ---------- Beca Andes Tech ----------
    {"id": "andestech-1", "beca": "Beca Andes Tech",
     "texto": "La Beca Andes Tech apoya maestrías y doctorados en ingeniería, ciencia de datos, inteligencia artificial, robótica y ciberseguridad. Incluye matrícula completa, una laptop de alto rendimiento y 1,500 soles mensuales para estudios en el Perú, o 1,600 dólares mensuales para estudios en el exterior."},
    {"id": "andestech-2", "beca": "Beca Andes Tech",
     "texto": "Los postulantes a la Beca Andes Tech deben pertenecer al tercio superior de su promoción, presentar un portafolio técnico (proyectos de código abierto, patentes o publicaciones) y aprobar una entrevista técnica en línea. La edad máxima es de 40 años. Las postulaciones se reciben del 1 de agosto al 15 de octubre."},

    # ---------- Beca Mujer Científica del Bicentenario ----------
    {"id": "mujer-1", "beca": "Beca Mujer Científica del Bicentenario",
     "texto": "La Beca Mujer Científica del Bicentenario promueve la participación de mujeres en ciencia y tecnología. Financia maestrías y doctorados en áreas STEM, y además cubre servicio de cuidado infantil durante los estudios con un monto de 600 soles mensuales por hijo menor de 6 años."},
    {"id": "mujer-2", "beca": "Beca Mujer Científica del Bicentenario",
     "texto": "Pueden postular a la Beca Mujer Científica del Bicentenario mujeres peruanas con grado de bachiller o magíster en ciencias, sin límite de edad. Se exige una carta de motivación, dos cartas de recomendación académicas y una constancia de admisión o de estar en proceso de admisión. Hay una cuota del 30% para postulantes de zonas rurales."},

    # ---------- Beca Docente Excelencia ----------
    {"id": "docente-1", "beca": "Beca Docente Excelencia",
     "texto": "La Beca Docente Excelencia es para profesores de colegios públicos que deseen cursar una maestría en educación, didáctica de las ciencias o gestión escolar. Se estudia en modalidad semipresencial en universidades peruanas, con matrícula cubierta al 100% y una licencia con goce de haber de 6 meses para elaborar la tesis."},
    {"id": "docente-2", "beca": "Beca Docente Excelencia",
     "texto": "Requisitos de la Beca Docente Excelencia: ser docente nombrado con al menos 5 años de servicio, haber obtenido una evaluación de desempeño destacada y contar con el visto bueno de la UGEL correspondiente. No se exige certificado de idiomas."},

    # ---------- Beca Salud Pública Nacional ----------
    {"id": "salud-1", "beca": "Beca Salud Pública Nacional",
     "texto": "La Beca Salud Pública Nacional financia maestrías y especialidades en epidemiología, salud pública, gestión hospitalaria y salud intercultural. Está dirigida a profesionales de la salud (médicos, enfermeras, obstetras, odontólogos) que trabajen en postas, centros de salud y hospitales del Ministerio de Salud (Minsa) ubicados fuera de Lima, incluidas zonas rurales. Cubre matrícula y una subvención de 2,000 soles al mes."},
    {"id": "salud-2", "beca": "Beca Salud Pública Nacional",
     "texto": "Para la Beca Salud Pública Nacional se necesita título profesional en ciencias de la salud, dos años de experiencia en el primer nivel de atención y un compromiso de permanencia de 3 años en la región de origen luego de culminar el posgrado. La convocatoria se abre en enero y cierra el 28 de febrero."},

    # ---------- Beca Pacífico Global (intercambio corto) ----------
    {"id": "pacifico-1", "beca": "Beca Pacífico Global",
     "texto": "La Beca Pacífico Global financia pasantías de investigación y estancias cortas de 3 a 6 meses en universidades de Asia-Pacífico para estudiantes de maestría o doctorado que ya estén matriculados en una universidad peruana. Cubre pasajes, alojamiento y un estipendio de 1,200 dólares mensuales."},

    # ---------- Documentos generales ----------
    {"id": "general-docs", "beca": "General",
     "texto": "Documentos comunes para postular a cualquier beca de INBECA: copia del DNI, grado de bachiller o título profesional, certificado de notas con promedio ponderado, currículum vitae documentado, carta de motivación de máximo 2 páginas, dos cartas de recomendación y declaración jurada de no tener deudas con el Estado ni otra beca vigente."},
    {"id": "general-proceso", "beca": "General",
     "texto": "El proceso de postulacióntiene cuatro etapas: registro en el portal de INBECA, evaluación documentaria, evaluación de mérito académico con entrevista y publicación de resultados. Los postulantes pueden presentar un reclamo dentro de los 5 días hábiles posteriores a la publicación de resultados."},
    {"id": "general-retorno", "beca": "General",
     "texto": "Los becarios de INBECA firman un contrato de retorno: deben regresar al Perú dentro de los 60 días posteriores a la culminación de sus estudios y prestar servicios en el país por un tiempo equivalente al doble del periodo financiado. El incumplimiento obliga a devolver el monto total de la beca con intereses."},
    {"id": "general-contacto", "beca": "General",
     "texto": "Contacto de INBECA (institución ficticia): correo informes@inbeca.example.pe, teléfono (01) 555-0100 y atención presencial de lunes a viernes de 9:00 a 17:00 horas en la Av. del Saber 123, Lima."},
]

BECAS = sorted({d["beca"] for d in KB if d["beca"] != "General"})
print(f"{len(KB)} fragmentos | {len(BECAS)} becas")

18 fragmentos | 7 becas


## 2. Componente 1 — Chatbot basado en **reglas**

Se normaliza el texto (minúsculas, sin tildes) y se compara con patrones. Es rápido, determinista y no consume LLM: ideal para lo predecible.

In [4]:
def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFD", texto.lower())
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", texto).strip()

def _listar_becas():
    return "Estas son las becas de posgrado que conozco:\n" + "\n".join(f"  • {b}" for b in BECAS) + \
           "\n\nPregúntame por requisitos, montos o fechas de cualquiera de ellas."

REGLAS = [
    ("saludo",   r"^(hola|buenas|buenos dias|buenas tardes|buenas noches|hey|que tal)\b",
        "¡Hola! Soy BecaBot. Te ayudo con información sobre becas de posgrado en el Perú. ¿Qué deseas saber?"),
    ("despedida", r"\b(adios|chau|chao|hasta luego|nos vemos|gracias,? eso es todo|salir)\b",
        "¡Hasta pronto! Mucho éxito en tu postulación."),
    ("gracias",  r"^(muchas )?gracias\b",
        "¡Con gusto! ¿Tienes otra consulta?"),
    ("listado",  r"\b(que|cuales) becas\b|\blista(r|do)? (de )?becas\b|\bbecas disponibles\b",
        _listar_becas),
    ("identidad", r"\b(quien eres|que eres|como te llamas)\b",
        "Soy BecaBot, un chatbot híbrido (reglas + embeddings + RAG + LLM local) sobre becas ficticias de posgrado."),
]
REGLAS_C = [(n, re.compile(p), r) for n, p, r in REGLAS]

def responder_reglas(pregunta: str):
    """Devuelve (nombre_regla, respuesta) o None si ninguna regla aplica."""
    q = normalizar(pregunta)
    for nombre, patron, resp in REGLAS_C:
        if patron.search(q):
            return nombre, (resp() if callable(resp) else resp)
    return None

for p in ["Hola!", "¿Qué becas hay?", "gracias", "¿Cuál es el monto de Andes Tech?"]:
    print(f"> {p}\n  {responder_reglas(p)}\n")

> Hola!
  ('saludo', '¡Hola! Soy BecaBot. Te ayudo con información sobre becas de posgrado en el Perú. ¿Qué deseas saber?')

> ¿Qué becas hay?
  ('listado', 'Estas son las becas de posgrado que conozco:\n  • Beca Amazonía Investigadora\n  • Beca Andes Tech\n  • Beca Docente Excelencia\n  • Beca Horizonte Perú\n  • Beca Mujer Científica del Bicentenario\n  • Beca Pacífico Global\n  • Beca Salud Pública Nacional\n\nPregúntame por requisitos, montos o fechas de cualquiera de ellas.')

> gracias
  ('gracias', '¡Con gusto! ¿Tienes otra consulta?')

> ¿Cuál es el monto de Andes Tech?
  None



## 3. Componente 2 — **NLP con embeddings** y similitud coseno

Cada fragmento se convierte en un vector denso; la consulta también. La similitud coseno es

$$\cos(\mathbf{q},\mathbf{d}) = \frac{\mathbf{q}\cdot\mathbf{d}}{\|\mathbf{q}\|\,\|\mathbf{d}\|}$$

Si normalizamos los vectores a norma 1, basta un producto punto.

In [5]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(EMB_MODEL)
textos = [d["texto"] for d in KB]
EMB_KB = encoder.encode(textos, normalize_embeddings=True)   # (n_docs, dim), norma 1
print("Matriz de embeddings:", EMB_KB.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Matriz de embeddings: (18, 384)


In [6]:
def similitud_coseno(a: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Coseno entre un vector `a` y cada fila de `B` (implementación explícita)."""
    return (B @ a) / (np.linalg.norm(B, axis=1) * np.linalg.norm(a) + 1e-12)

def buscar(consulta: str, k: int = TOP_K):
    q = encoder.encode([consulta], normalize_embeddings=True)[0]
    scores = similitud_coseno(q, EMB_KB)
    idx = np.argsort(-scores)[:k]
    return [(KB[i], float(scores[i])) for i in idx]

def mostrar(consulta, k=TOP_K):
    print(f"Consulta: {consulta}")
    for doc, s in buscar(consulta, k):
        print(f"   {s:.3f} | {doc['id']:<18} | {doc['texto'][:85]}...")
    print()

mostrar("¿Cuánto me pagan al mes si estudio inteligencia artificial?")
mostrar("Soy madre y quiero hacer un doctorado en ciencias")
mostrar("¿Qué pasa si no vuelvo al país después de estudiar?")
mostrar("Trabajo de profesor en un colegio estatal, ¿hay apoyo?")

Consulta: ¿Cuánto me pagan al mes si estudio inteligencia artificial?
   0.424 | andestech-1        | La Beca Andes Tech apoya maestrías y doctorados en ingeniería, ciencia de datos, inte...
   0.359 | amazonia-2         | Para postular a la Beca Amazonía Investigadora se requiere grado de magíster, un proy...
   0.348 | amazonia-1         | La Beca Amazonía Investigadora está dirigida a estudiantes de doctorado que investigu...

Consulta: Soy madre y quiero hacer un doctorado en ciencias
   0.575 | mujer-1            | La Beca Mujer Científica del Bicentenario promueve la participación de mujeres en cie...
   0.474 | salud-2            | Para la Beca Salud Pública Nacional se necesita título profesional en ciencias de la ...
   0.392 | mujer-2            | Pueden postular a la Beca Mujer Científica del Bicentenario mujeres peruanas con grad...

Consulta: ¿Qué pasa si no vuelvo al país después de estudiar?
   0.409 | general-retorno    | Los becarios de INBECA firman un contrato de ret

Observa que las consultas **no comparten palabras exactas** con los textos ("madre" ↔ "cuidado infantil", "no vuelvo" ↔ "retorno"): eso es la ventaja de la búsqueda semántica frente a la de palabras clave.

## 4. Componente 3 — **RAG** (Retrieval-Augmented Generation)

Se arma un prompt con los fragmentos recuperados y se instruye al modelo a responder **solo** con ese contexto.

In [7]:
SYSTEM_PROMPT = (
    "Eres BecaBot, asistente sobre becas de posgrado en el Perú. "
    "Responde SIEMPRE en español, de forma breve y clara (máximo 4 oraciones). "
    "Usa ÚNICAMENTE la información del CONTEXTO. Solo si el contexto no tiene nada relacionado "
    "con la pregunta, di que no tienes esa información. No inventes datos."
)

def construir_contexto(resultados):
    return "\n".join(f"[{i+1}] ({d['beca']}) {d['texto']}" for i, (d, _) in enumerate(resultados))

def construir_mensajes(pregunta, resultados, historial=None):
    contexto = construir_contexto(resultados)
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    msgs += (historial or [])[-4:]               # memoria corta: últimos 2 turnos
    msgs.append({"role": "user",
                 "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"})
    return msgs

res = buscar("¿Cuántos años debo trabajar en Perú tras la Beca Horizonte?")
print(construir_mensajes("¿Cuántos años debo trabajar en Perú tras la Beca Horizonte?", res)[-1]["content"])

CONTEXTO:
[1] (Beca Horizonte Perú) Requisitos de la Beca Horizonte Perú: ser peruano, tener grado de bachiller con promedio ponderado mínimo de 14 sobre 20, contar con carta de admisión de la universidad de destino, certificado de inglés nivel B2 y no superar los 35 años de edad. Los becarios deben retornar al Perú y trabajar 2 años por cada año de beca.
[2] (Beca Horizonte Perú) La Beca Horizonte Perú financia estudios de maestría en universidades del extranjero ubicadas en el top 200 de un ranking internacional. Cubre matrícula, pensión, pasaje de ida y vuelta, seguro médico y una subvención mensual de 1,800 dólares. La duración máxima es de 24 meses.
[3] (General) Los becarios de INBECA firman un contrato de retorno: deben regresar al Perú dentro de los 60 días posteriores a la culminación de sus estudios y prestar servicios en el país por un tiempo equivalente al doble del periodo financiado. El incumplimiento obliga a devolver el monto total de la beca con intereses.

PREGUNTA: ¿

## 5. Componente 4 — **LLM local con Ollama**

`qwen2.5:1.5b` ocupa ~1 GB en disco y ~2 GB de RAM, y corre en CPU. Se consume vía la API REST local (`/api/chat`).

In [8]:
def ollama_disponible(modelo: str = LLM_MODEL) -> bool:
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
        nombres = [m["name"] for m in r.json().get("models", [])]
        ok = any(n == modelo or n.startswith(modelo + ":") for n in nombres)
        if not ok:
            print(f"AVISO: Ollama responde, pero falta el modelo. Ejecuta: ollama pull {modelo}")
        return ok
    except requests.RequestException:
        print("AVISO: No se pudo conectar con Ollama. ¿Está instalado y ejecutándose? (ollama serve)")
        return False

def llm_chat(mensajes, modelo=LLM_MODEL, temperatura=0.2) -> str:
    r = requests.post(f"{OLLAMA_URL}/api/chat", timeout=180, json={
        "model": modelo, "messages": mensajes, "stream": False,
        "options": {"temperature": temperatura, "num_ctx": 2048, "num_predict": 256},
    })
    r.raise_for_status()
    return r.json()["message"]["content"].strip()

LLM_OK = ollama_disponible()
print("LLM disponible:", LLM_OK)
if LLM_OK:
    print(llm_chat([{"role": "user", "content": "Saluda en una frase corta en español."}]))

LLM disponible: True


¡Hola! ¿Cómo estás?


## 6. Chatbot híbrido: reglas → embeddings → RAG → LLM

Si Ollama no está disponible, el bot degrada con elegancia a una **respuesta extractiva** (devuelve el fragmento más relevante).

In [9]:
class BecaBot:
    def __init__(self, k=TOP_K, umbral=UMBRAL, usar_llm=True):
        self.k, self.umbral, self.usar_llm = k, umbral, usar_llm
        self.historial = []

    def responder(self, pregunta: str, debug=False) -> str:
        # 1) Reglas
        r = responder_reglas(pregunta)
        if r:
            nombre, texto = r
            return f"[regla:{nombre}] {texto}" if debug else texto

        # 2) Búsqueda semántica (embeddings + coseno)
        resultados = buscar(pregunta, self.k)
        mejor = resultados[0][1]
        if mejor < self.umbral:
            msg = ("No encontré información sobre eso en mi base de datos. "
                   "Puedes preguntarme por requisitos, montos, fechas o documentos de las becas disponibles.")
            return f"[sin-contexto score={mejor:.2f}] {msg}" if debug else msg

        # 3) RAG + 4) LLM
        # además del umbral absoluto, descarta fragmentos muy por debajo del mejor (evita ruido en el contexto)
        relevantes = [(d, s) for d, s in resultados if s >= self.umbral and s >= mejor - 0.25]
        if self.usar_llm and LLM_OK:
            try:
                resp = llm_chat(construir_mensajes(pregunta, relevantes, self.historial))
                modo = "rag+llm"
            except requests.RequestException as e:
                resp, modo = f"(LLM no respondió: {e}) {relevantes[0][0]['texto']}", "extractivo"
        else:
            resp, modo = relevantes[0][0]["texto"], "extractivo"

        self.historial += [{"role": "user", "content": pregunta},
                           {"role": "assistant", "content": resp}]
        if debug:
            fuentes = ", ".join(f"{d['id']}({s:.2f})" for d, s in relevantes)
            return f"[{modo}] {resp}\n      ↳ fuentes: {fuentes}"
        return resp

bot = BecaBot()
preguntas = [
    "Hola",
    "¿Qué becas hay?",
    "¿Cuánto paga la beca Andes Tech y hasta qué edad puedo postular?",
    "Soy enfermera en una posta rural, ¿qué beca me conviene?",
    "¿Qué documentos necesito presentar?",
    "¿Cuál es la capital de Francia?",
    "Gracias, adiós",
]
for p in preguntas:
    print(f"Usuario: {p}\nBot: {bot.responder(p, debug=True)}\n")

Usuario: Hola
Bot: [regla:saludo] ¡Hola! Soy BecaBot. Te ayudo con información sobre becas de posgrado en el Perú. ¿Qué deseas saber?

Usuario: ¿Qué becas hay?
Bot: [regla:listado] Estas son las becas de posgrado que conozco:
  • Beca Amazonía Investigadora
  • Beca Andes Tech
  • Beca Docente Excelencia
  • Beca Horizonte Perú
  • Beca Mujer Científica del Bicentenario
  • Beca Pacífico Global
  • Beca Salud Pública Nacional

Pregúntame por requisitos, montos o fechas de cualquiera de ellas.



Usuario: ¿Cuánto paga la beca Andes Tech y hasta qué edad puedo postular?
Bot: [rag+llm] La beca Andes Tech cubre matrícula completa, una laptop de alto rendimiento y 1,500 soles mensuales para estudios en el Perú, o 1,600 dólares mensuales para estudios en el exterior. La edad máxima para postular es de 40 años.
      ↳ fuentes: andestech-2(0.74), andestech-1(0.66), horizonte-2(0.54)



Usuario: Soy enfermera en una posta rural, ¿qué beca me conviene?
Bot: [rag+llm] La beca Mujer Científica del Bicentenario podría ser la más adecuada para usted, ya que se dirige a mujeres con grado de bachiller o magíster en ciencias, sin límite de edad.
      ↳ fuentes: salud-1(0.39), mujer-2(0.33)



Usuario: ¿Qué documentos necesito presentar?
Bot: [rag+llm] Para postular a cualquier beca de INBECA, necesitará los siguientes documentos:

1. Copia del DNI
2. Grado de bachiller o título profesional
3. Certificado de notas con promedio ponderado
4. Currículum vitae documentado
5. Carta de motivación de máximo 2 páginas
6. Dos cartas de recomendación
7. Declaración jurada de no tener deudas con el Estado ni otra beca vigente.
      ↳ fuentes: general-docs(0.62), general-proceso(0.40)

Usuario: ¿Cuál es la capital de Francia?
Bot: [sin-contexto score=0.18] No encontré información sobre eso en mi base de datos. Puedes preguntarme por requisitos, montos, fechas o documentos de las becas disponibles.

Usuario: Gracias, adiós
Bot: [regla:despedida] ¡Hasta pronto! Mucho éxito en tu postulación.



## 7. Evaluación rápida de la recuperación

Mini conjunto de prueba (consulta → fragmento esperado) para medir **Hit@1** y **Hit@3** del buscador semántico.

In [10]:
PRUEBAS = [
    ("¿Cuánto me dan al mes para estudiar IA en el extranjero?", "andestech-1"),
    ("¿Hasta cuándo puedo postular a la beca de ingeniería y robótica?", "andestech-2"),
    ("Quiero hacer un doctorado sobre bosques amazónicos", "amazonia-1"),
    ("¿Hay ayuda para el cuidado de mis hijos mientras estudio?", "mujer-1"),
    ("Soy profesor de colegio público y quiero una maestría en educación", "docente-1"),
    ("¿Qué nivel de inglés piden para la maestría en el exterior?", "horizonte-2"),
    ("¿Cuándo salen los resultados de Horizonte?", "horizonte-3"),
    ("Soy médico del Minsa fuera de Lima, ¿qué beca hay?", "salud-1"),
    ("¿Se puede hacer una pasantía corta en Asia?", "pacifico-1"),
    ("¿Cuántos días tengo para reclamar si no gano?", "general-proceso"),
    ("¿Qué ocurre si no regreso al Perú al terminar?", "general-retorno"),
    ("¿A qué correo escribo para pedir información?", "general-contacto"),
]

hit1 = hit3 = 0
for q, esperado in PRUEBAS:
    ids = [d["id"] for d, _ in buscar(q, 3)]
    hit1 += ids[0] == esperado
    hit3 += esperado in ids
    print(("[OK]    " if ids[0] == esperado else "[TOP-3] " if esperado in ids else "[FALLA] "), q, "→", ids[0])
print(f"\nHit@1 = {hit1/len(PRUEBAS):.0%}   Hit@3 = {hit3/len(PRUEBAS):.0%}")

[TOP-3]  ¿Cuánto me dan al mes para estudiar IA en el extranjero? → pacifico-1
[OK]     ¿Hasta cuándo puedo postular a la beca de ingeniería y robótica? → andestech-2
[OK]     Quiero hacer un doctorado sobre bosques amazónicos → amazonia-1
[OK]     ¿Hay ayuda para el cuidado de mis hijos mientras estudio? → mujer-1


[OK]     Soy profesor de colegio público y quiero una maestría en educación → docente-1
[TOP-3]  ¿Qué nivel de inglés piden para la maestría en el exterior? → docente-2
[OK]     ¿Cuándo salen los resultados de Horizonte? → horizonte-3
[OK]     Soy médico del Minsa fuera de Lima, ¿qué beca hay? → salud-1


[OK]     ¿Se puede hacer una pasantía corta en Asia? → pacifico-1
[OK]     ¿Cuántos días tengo para reclamar si no gano? → general-proceso
[OK]     ¿Qué ocurre si no regreso al Perú al terminar? → general-retorno
[OK]     ¿A qué correo escribo para pedir información? → general-contacto

Hit@1 = 83%   Hit@3 = 100%


## 8. Conversación interactiva

Escribe `salir` para terminar.

In [11]:
def chat():
    bot = BecaBot()
    print("BecaBot — escribe 'salir' para terminar.\n")
    while True:
        try:
            q = input("Tú: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not q:
            continue
        print("Bot:", bot.responder(q, debug=True), "\n")
        if re.search(r"\b(salir|adios|chao)\b", normalizar(q)):
            break

# chat()   # <- descomenta para conversar

## 9. Conclusiones y mejoras posibles

- Las **reglas** resuelven lo trivial sin costo; los **embeddings** capturan la intención aunque cambien las palabras; **RAG** limita al LLM a información verificable; el **LLM pequeño** solo redacta.
- El umbral de coseno evita que el bot "alucine" ante preguntas fuera de dominio.
- Mejoras: *chunking* automático de PDFs, reranking con un cross-encoder, búsqueda híbrida (BM25 + embeddings), `ollama` con `stream=True`, persistir embeddings en FAISS/Chroma, evaluar con más preguntas.